In [5]:
# 1. IMPORTS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_squared_error
import pickle

# 2. LOAD DATA
try:
    df = pd.read_csv('weight-height.csv')
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Error: 'weight-height.csv' not found. Please upload the file.")
    # Creating dummy data just so the code doesn't crash if you run it without the file
    data = {
        'Gender': ['Male']*500 + ['Female']*500,
        'Height': np.random.normal(69, 3, 1000), # Inches
        'Weight': np.random.normal(180, 20, 1000) # Pounds
    }
    df = pd.DataFrame(data)

# 3. DATA PREPROCESSING (Unit Conversion & Encoding)

print("Converting units to Metric (cm/kg)...")
df['Height'] = df['Height'] * 2.54
df['Weight'] = df['Weight'] * 0.453592

# Male = 1, Female = 0
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})

# 4. DEFINE FEATURES AND TARGET
X = df[['Weight', 'Gender']]
y = df['Height']

# 5. TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. CREATE PIPELINE (Polynomial -> Scaling -> Regression)
pipeline = make_pipeline(
    PolynomialFeatures(degree=2), 
    StandardScaler(),             
    LinearRegression()           
)

# 7. TRAIN MODEL
print("Training the model...")
pipeline.fit(X_train, y_train)

# 8. EVALUATE MODEL
y_pred = pipeline.predict(X_test)
r2 = r2_score(y_test, y_pred)
print("="*30)
print(f"Model R2 Score: {r2*100:.2f}%")
print("="*30)

# 9. SAVE MODEL (PICKLE)
with open('height_predictor_pipeline.pkl', 'wb') as f:
    pickle.dump(pipeline, f)
print("Model pipeline saved as 'height_predictor_pipeline.pkl'")

# --- TEST PREDICTION ---
test_weight = 80 # kg
test_gender = 1  # Male
input_data = pd.DataFrame([[test_weight, test_gender]], columns=['Weight', 'Gender'])

predicted_height = pipeline.predict(input_data)[0]
print(f"\nTest Prediction for 80kg Male:")
print(f"Predicted Height: {predicted_height:.2f} cm")

Dataset loaded successfully.
Converting units to Metric (cm/kg)...
Training the model...
Model R2 Score: 86.24%
Model pipeline saved as 'height_predictor_pipeline.pkl'

Test Prediction for 80kg Male:
Predicted Height: 171.98 cm
